# 01 · pass@k: the unbiased estimator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/evals/01_pass_at_k.ipynb) &nbsp; 📖 Concept deck: [pass@k and sampling metrics](https://nilesh-salpe.github.io/visual-ai/evals/03-pass-at-k/) &nbsp; ⏱️ Suggested time: **30 min**

**You will build:** the Codex paper's unbiased pass@k estimator and its numerically stable form, and a simulation showing why the obvious estimator is biased.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def sigmoid(z):
    return 1 / (1 + np.exp(-z))

from math import comb

## Task 1: The unbiased estimator

pass@k = probability that **at least one of k** samples passes the unit tests. Sampling exactly k per problem is noisy, so Chen et al. (2021) sample **n ≥ k**, count the **c** that pass, and estimate
$$\text{pass@}k = 1 - \frac{\binom{n-c}{k}}{\binom{n}{k}}$$
i.e. one minus the chance that a random size-k subset of the n samples contains no passing sample. If $n - c < k$, every subset contains a pass, so the answer is 1.
Binomials overflow for large n, so use the paper's stable product form: $1 - \prod_{i=n-c+1}^{n}\left(1 - \frac{k}{i}\right)$.

In [ ]:
def pass_at_k(n, c, k):
    # TODO: return 1.0 if n - c < k; else 1 - prod(1 - k / arange(n - c + 1, n + 1))
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_pass_at_k(f):
    for n, c, k in [(10, 3, 1), (10, 3, 5), (20, 0, 5), (20, 20, 1), (200, 7, 100), (5, 2, 4)]:
        want = 1.0 if n - c < k else 1 - comb(n - c, k) / comb(n, k)
        assert np.isclose(f(n, c, k), want), f"pass_at_k({n},{c},{k}) = {f(n, c, k)}, expected {want}"
    assert np.isclose(f(10, 3, 1), 0.3), "pass@1 must equal c/n"
    assert np.isfinite(f(5000, 30, 1000)), "must not overflow for large n"

check("pass_at_k", _test_pass_at_k, pass_at_k)

## Task 2: The tempting (biased) estimator

The obvious shortcut: estimate the per-sample pass rate $\hat p = c/n$, then use $1 - (1 - \hat p)^k$.
It looks right, because that is the formula for the true pass@k given the true p. But $1-(1-p)^k$ is concave in p, so plugging in a noisy $\hat p$ **underestimates** on average (Jensen's inequality). The Codex paper notes this estimator is biased.

In [ ]:
def naive_pass_at_k(n, c, k):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_naive_pass_at_k(f):
    assert np.isclose(f(10, 3, 1), 0.3)
    assert np.isclose(f(10, 3, 5), 1 - 0.7 ** 5)

check("naive_pass_at_k", _test_naive_pass_at_k, naive_pass_at_k)

## Task 3: Measure the bias by simulation

For a problem whose true per-sample pass rate is `p`, the true pass@k is $1-(1-p)^k$. Simulate `trials` evaluations: each draws $c \sim \mathrm{Binomial}(n, p)$ and applies an estimator. Return `(mean_estimate - true_value)` for the given estimator function.
The tests check the unbiased estimator's mean error is ≈ 0 and the naive one is clearly negative for small p.

In [ ]:
def estimator_bias(estimator, p, n, k, trials=20000, seed=0):
    rng = np.random.default_rng(seed)
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_estimator_bias(f):
    u = f(pass_at_k_solution if f.__name__.endswith('_solution') else _ref_unbiased, 0.05, 20, 10)
    nv = f(lambda n, c, k: 1 - (1 - c / n) ** k, 0.05, 20, 10)
    assert abs(u) < 0.01, f"unbiased estimator should have ~0 bias, got {u:+.4f}"
    assert nv < -0.03, f"naive estimator should underestimate here, got {nv:+.4f}"

def _ref_unbiased(n, c, k):
    return 1.0 if n - c < k else 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))

check("estimator_bias", _test_estimator_bias, estimator_bias)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: The unbiased estimator
def pass_at_k_solution(n, c, k):
    if n - c < k:
        return 1.0
    return 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))

check("pass_at_k (reference)", _test_pass_at_k, pass_at_k_solution)

In [ ]:
# @title Solution for Task 2: The tempting (biased) estimator
def naive_pass_at_k_solution(n, c, k):
    return 1 - (1 - c / n) ** k

check("naive_pass_at_k (reference)", _test_naive_pass_at_k, naive_pass_at_k_solution)

In [ ]:
# @title Solution for Task 3: Measure the bias by simulation
def estimator_bias_solution(estimator, p, n, k, trials=20000, seed=0):
    rng = np.random.default_rng(seed)
    cs = rng.binomial(n, p, trials)
    return np.mean([estimator(n, int(c), k) for c in cs]) - (1 - (1 - p) ** k)

check("estimator_bias (reference)", _test_estimator_bias, estimator_bias_solution)

---
## Experiments
**pass@1 vs pass@k on a benchmark.** Give 164 problems (HumanEval's size) a spread of difficulties, sample n = 200 per problem, and report pass@1, @10 and @100. Codex-12B scored 28.8% at pass@1 and 70.2% with 100 samples: sampling many attempts and picking any that passes is very effective when tests are available. Then the naive estimator's bias across p.

In [ ]:
rng = np.random.default_rng(0); p = rng.beta(0.4, 0.8, 164); n = 200
c = rng.binomial(n, p)
for k in [1, 10, 100]:
    print(f"pass@{k:<3} unbiased {np.mean([pass_at_k_solution(n, ci, k) for ci in c]):.3f}   naive {np.mean([naive_pass_at_k_solution(n, ci, k) for ci in c]):.3f}")
print("\nbias of the naive estimator (n=20, k=10):")
for pp in [0.01, 0.05, 0.1, 0.3]:
    print(f"  p={pp:<5} naive {estimator_bias_solution(naive_pass_at_k_solution, pp, 20, 10):+.4f}   unbiased {estimator_bias_solution(pass_at_k_solution, pp, 20, 10):+.4f}")

---
## Takeaways
* pass@k asks "does at least one of k attempts succeed?". It rewards coverage, not reliability, so always say which k (and which n) you report.
* Estimate it from n ≥ k samples with $1 - \binom{n-c}{k}/\binom{n}{k}$ (product form for stability). Plugging c/n into $1-(1-p)^k$ underestimates.
* pass@1 with many samples is just the average pass rate, with much lower variance than one sample per problem.